# 🎵 음악 장르 분류 & 추천 알고리즘
**Librosa 기반 오디오 특성 추출 → XGBoost 분류 → 코사인 유사도 추천**

사용 데이터셋: [GTZAN Dataset (Kaggle)](https://www.kaggle.com/datasets/andradaolteanu/gtzan-dataset-music-genre-classification)

---
### 목차
1. 환경 설정 및 데이터 다운로드
2. 오디오 파일 이해하기
3. 오디오 특성 추출
4. 음악 장르 분류 알고리즘 (XGBoost)
5. 음악 추천 알고리즘 (Cosine Similarity)

## 0. 환경 설정

In [ ]:
# 필요한 라이브러리 설치
!pip install librosa xgboost scikit-learn pandas numpy matplotlib seaborn ipython -q

## 1. 데이터셋 다운로드 (Kaggle API 방식 - Colab 기준)

> 로컬 환경이라면 [Kaggle 페이지](https://www.kaggle.com/datasets/andradaolteanu/gtzan-dataset-music-genre-classification)에서 직접 다운로드 후 `Data/` 폴더에 압축 해제하세요.

In [ ]:
import os

# ── Google Colab 환경일 때만 실행 ──
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install kaggle -q
    from google.colab import files
    print("kaggle.json 파일을 업로드하세요.")
    files.upload()

    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/kaggle.json
    !chmod 600 ~/.kaggle/kaggle.json

    !kaggle datasets download -d andradaolteanu/gtzan-dataset-music-genre-classification
    !unzip -q gtzan-dataset-music-genre-classification.zip -d Data
    DATA_PATH = 'Data'
else:
    # 로컬 환경: 데이터가 이미 있다고 가정
    DATA_PATH = 'Data'
    if not os.path.exists(DATA_PATH):
        print("⚠️  'Data/' 폴더가 없습니다. Kaggle에서 데이터셋을 다운로드한 뒤 압축을 해제해 주세요.")
        print("   https://www.kaggle.com/datasets/andradaolteanu/gtzan-dataset-music-genre-classification")

print(f"DATA_PATH = '{DATA_PATH}'")

## 2. 오디오 파일 이해하기

In [ ]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

SAMPLE_FILE = os.path.join(DATA_PATH, 'genres_original/reggae/reggae.00036.wav')

# 오디오 로드
y, sr = librosa.load(SAMPLE_FILE)

print(f"y (진폭 배열) : {y[:10]}  ...")
print(f"배열 길이     : {len(y):,}")
print(f"Sampling Rate : {sr:,} Hz")
print(f"음악 길이      : {len(y)/sr:.2f} 초")

In [ ]:
# 음악 재생 (Colab / Jupyter 에서 동작)
ipd.Audio(y, rate=sr)

### 2-1. 2D 음파 그래프 (Waveform)

In [ ]:
plt.figure(figsize=(16, 4))
librosa.display.waveshow(y=y, sr=sr)
plt.title('Waveform - reggae.00036')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()

### 2-2. Short-Time Fourier Transform (STFT)

In [ ]:
# STFT: 시간 영역 → 주파수 영역 변환
# n_fft: 한 번 FFT할 window size
# hop_length: 얼마나 이동하며 분석할지 (시간 해상도)
D = np.abs(librosa.stft(y, n_fft=2048, hop_length=512))
print(f"STFT shape: {D.shape}  (주파수 bin × 시간 프레임)")

plt.figure(figsize=(16, 4))
plt.plot(D[:, :100].T)   # 처음 100 프레임만 시각화
plt.title('STFT Magnitude (first 100 frames)')
plt.tight_layout()
plt.show()

### 2-3. Spectrogram

In [ ]:
DB = librosa.amplitude_to_db(D, ref=np.max)

plt.figure(figsize=(16, 5))
librosa.display.specshow(DB, sr=sr, hop_length=512, x_axis='time', y_axis='log')
plt.colorbar(format='%+2.0f dB')
plt.title('Spectrogram (dB) - reggae.00036')
plt.tight_layout()
plt.show()

### 2-4. Mel Spectrogram

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Reggae
y_r, sr_r = librosa.load(os.path.join(DATA_PATH, 'genres_original/reggae/reggae.00036.wav'))
S_r = librosa.feature.melspectrogram(y=y_r, sr=sr_r)
S_r_db = librosa.amplitude_to_db(S_r, ref=np.max)
librosa.display.specshow(S_r_db, sr=sr_r, hop_length=512, x_axis='time', y_axis='log', ax=axes[0])
axes[0].set_title('Mel Spectrogram - Reggae')

# Classical
y_c, sr_c = librosa.load(os.path.join(DATA_PATH, 'genres_original/classical/classical.00036.wav'))
y_c, _ = librosa.effects.trim(y_c)
S_c = librosa.feature.melspectrogram(y=y_c, sr=sr_c)
S_c_db = librosa.amplitude_to_db(S_c, ref=np.max)
librosa.display.specshow(S_c_db, sr=sr_c, hop_length=512, x_axis='time', y_axis='log', ax=axes[1])
axes[1].set_title('Mel Spectrogram - Classical')

plt.tight_layout()
plt.show()

## 3. 오디오 특성 추출 (Audio Feature Extraction)

| 특성 | 설명 |
|------|------|
| Tempo (BPM) | 음악의 빠르기 |
| Zero Crossing Rate | 음파가 양↔음으로 바뀌는 비율 |
| Harmonic / Percussive | 음색(색깔) / 리듬 성분 분리 |
| Spectral Centroid | 주파수 무게 중심 |
| Spectral Rolloff | 저주파 에너지 집중도 |
| MFCCs | 소리 고유 특징 벡터 (10-20차원) |
| Chroma Frequencies | 음계(옥타브) 특성, 화음 인식 |

In [ ]:
# 샘플 파일 다시 로드 (classical 으로 비교)
SAMPLE_FILE = os.path.join(DATA_PATH, 'genres_original/reggae/reggae.00036.wav')
y, sr = librosa.load(SAMPLE_FILE)

# ── 3-1. Tempo ──
tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
print(f"Tempo: {float(tempo):.1f} BPM")

In [ ]:
# ── 3-2. Zero Crossing Rate ──
zcr = librosa.zero_crossings(y, pad=False)
print(f"Zero Crossing 총 횟수: {sum(zcr):,}")

# 구간 확대 시각화
n0, n1 = 9000, 9040
plt.figure(figsize=(14, 3))
plt.plot(y[n0:n1])
plt.axhline(0, color='r', linewidth=0.8, linestyle='--')
plt.title(f'Zero Crossing Rate (sample {n0}~{n1})')
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"해당 구간 ZCR: {sum(librosa.zero_crossings(y[n0:n1], pad=False))}")

In [ ]:
# ── 3-3. Harmonic & Percussive ──
y_harm, y_perc = librosa.effects.hpss(y)

plt.figure(figsize=(14, 3))
plt.plot(y_harm, color='b', alpha=0.7, label='Harmonic')
plt.plot(y_perc, color='r', alpha=0.7, label='Percussive')
plt.legend()
plt.title('Harmonic vs Percussive')
plt.tight_layout()
plt.show()

In [ ]:
import sklearn
import sklearn.preprocessing

def normalize(x, axis=0):
    return sklearn.preprocessing.minmax_scale(x, axis=axis)

frames = range(len(librosa.feature.spectral_centroid(y=y, sr=sr)[0]))
t = librosa.frames_to_time(frames)

# ── 3-4. Spectral Centroid ──
spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr)[0]

plt.figure(figsize=(14, 3))
librosa.display.waveshow(y, sr=sr, alpha=0.4, color='b')
plt.plot(t, normalize(spectral_centroids), color='r', label='Spectral Centroid')
plt.legend()
plt.title('Spectral Centroid')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3-5. Spectral Rolloff ──
spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]

plt.figure(figsize=(14, 3))
librosa.display.waveshow(y, sr=sr, alpha=0.4, color='b')
plt.plot(t, normalize(spectral_rolloff), color='r', label='Spectral Rolloff')
plt.legend()
plt.title('Spectral Rolloff (85% energy threshold)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3-6. MFCCs ──
mfccs = librosa.feature.mfcc(y=y, sr=sr)
mfccs_norm = normalize(mfccs, axis=1)

print(f"MFCC shape: {mfccs.shape}")
print(f"mean: {mfccs_norm.mean():.4f} | var: {mfccs_norm.var():.4f}")

plt.figure(figsize=(14, 4))
librosa.display.specshow(mfccs_norm, sr=sr, x_axis='time')
plt.colorbar()
plt.title('MFCCs (normalized)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3-7. Chroma Frequencies ──
chromagram = librosa.feature.chroma_stft(y=y, sr=sr, hop_length=512)

plt.figure(figsize=(14, 4))
librosa.display.specshow(chromagram, x_axis='time', y_axis='chroma', hop_length=512)
plt.colorbar()
plt.title('Chroma Frequencies')
plt.tight_layout()
plt.show()

## 4. 음악 장르 분류 알고리즘 (XGBoost)

GTZAN 데이터셋에서 미리 추출된 `features_3_sec.csv`를 활용합니다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix
from xgboost import XGBClassifier
import seaborn as sns

# ── 4-1. 데이터 로드 ──
CSV_PATH = os.path.join(DATA_PATH, 'features_3_sec.csv')
df = pd.read_csv(CSV_PATH)
print(f"데이터 크기: {df.shape}")
print(f"장르 목록: {sorted(df['label'].unique())}")
df.head(3)

In [ ]:
# ── 4-2. 전처리 ──
X = df.drop(columns=['filename', 'length', 'label'])
y_raw = df['label']

# 레이블 인코딩 (범주형 → 수치형)
le = LabelEncoder()
y_enc = le.fit_transform(y_raw)
print(f"레이블 인코딩: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# MinMax 스케일링 (0~1)
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_scaled.head(3)

In [ ]:
# ── 4-3. 데이터셋 분할 ──
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_enc, test_size=0.2, random_state=2021, stratify=y_enc
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

In [ ]:
# ── 4-4. XGBoost 학습 ──
# XGBoost: 잔차 기반 트리를 반복 추가해 약한 학습기를 강력한 학습기로 변환
xgb = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)
xgb.fit(X_train, y_train, verbose=False)

y_preds = xgb.predict(X_test)
acc = accuracy_score(y_test, y_preds)
print(f"✅ Accuracy: {acc:.4f} ({acc*100:.2f}%)")

In [ ]:
# ── 4-5. Confusion Matrix ──
genre_labels = le.classes_
cm = confusion_matrix(y_test, y_preds)

plt.figure(figsize=(12, 9))
sns.heatmap(
    cm,
    annot=True, fmt='d',
    xticklabels=genre_labels,
    yticklabels=genre_labels,
    cmap='Blues'
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix (Accuracy: {acc*100:.2f}%)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4-6. Feature Importance ──
fi = pd.Series(xgb.feature_importances_, index=X_scaled.columns)
top_features = fi.nlargest(15)

plt.figure(figsize=(10, 5))
top_features.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print("\nTop 5 중요 특성:")
for feat, imp in top_features.head(5).items():
    print(f"  {feat}: {imp:.4f}")

## 5. 음악 추천 알고리즘 (Cosine Similarity)

오디오 특성 벡터 간 코사인 유사도를 계산하여 유사한 노래를 추천합니다.

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}$$

> cos θ = 1 이면 완전 동일, cos θ = -1 이면 완전 반대

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import scale

# ── 5-1. 30초 특성 데이터 로드 ──
CSV_30 = os.path.join(DATA_PATH, 'features_30_sec.csv')
df_30 = pd.read_csv(CSV_30, index_col='filename')

labels_df = df_30[['label']].copy()
df_30 = df_30.drop(columns=['length', 'label'])

# 표준화 (평균 0, 표준편차 1)
df_30_scaled = pd.DataFrame(scale(df_30), index=df_30.index, columns=df_30.columns)
print(f"30초 데이터: {df_30_scaled.shape}")
df_30_scaled.head(3)

In [ ]:
# ── 5-2. 코사인 유사도 행렬 계산 ──
similarity = cosine_similarity(df_30_scaled)
sim_df = pd.DataFrame(similarity, index=df_30_scaled.index, columns=df_30_scaled.index)
print(f"유사도 행렬 shape: {sim_df.shape}")

In [ ]:
# ── 5-3. 추천 함수 ──
def recommend_songs(song_name: str, n: int = 5) -> pd.DataFrame:
    """
    입력 곡과 가장 유사한 n개의 노래를 추천합니다.
    
    Parameters
    ----------
    song_name : str  파일명 (예: 'rock.00000.wav')
    n         : int  추천 개수 (기본값: 5)
    
    Returns
    -------
    DataFrame  곡명, 유사도, 장르 포함
    """
    if song_name not in sim_df.index:
        raise ValueError(f"'{song_name}' 를 찾을 수 없습니다.")
    
    series = sim_df[song_name].sort_values(ascending=False)
    series = series.drop(song_name)        # 자기 자신 제외
    top_n = series.head(n).reset_index()
    top_n.columns = ['song', 'similarity']
    top_n['genre'] = top_n['song'].apply(
        lambda s: labels_df.loc[s, 'label'] if s in labels_df.index else 'unknown'
    )
    return top_n

# 테스트
query = 'rock.00000.wav'
result = recommend_songs(query, n=5)
print(f"🎵 '{query}' 와 가장 유사한 노래 TOP 5:\n")
print(result.to_string(index=False))

In [ ]:
# 다른 장르로도 테스트
for test_song in ['jazz.00010.wav', 'classical.00005.wav', 'hiphop.00020.wav']:
    try:
        res = recommend_songs(test_song, n=3)
        print(f"\n🎵 '{test_song}' 추천:")
        print(res.to_string(index=False))
    except ValueError as e:
        print(e)

---
## 결과 요약

| 항목 | 결과 |
|------|------|
| 모델 | XGBoost (n_estimators=1000, lr=0.05) |
| 데이터셋 | GTZAN (10장르 × 100곡) |
| 분류 정확도 | ~88% |
| 추천 방식 | Cosine Similarity on 30-sec features |

### 핵심 인사이트
- 음성 데이터는 **최대한 많은 특징을 추출**해 모델에 입력하면 분류 성능이 향상됩니다.
- MFCCs가 전반적으로 가장 중요한 특성이며, `perceptr_var` 등 지각 관련 특성도 유효합니다.
- Rock ↔ Country 처럼 음악적 특성이 유사한 장르 간 혼동이 주요 오류입니다.